# FEniCSx Spielplatz

Hier können Sie mit FEniCSx experimentieren!

In [ ]:
# FEniCSx direkt importieren - Conda-Umgebung wird automatisch verwendet
# Prüfe, ob wir in der richtigen Umgebung sind
import sys
import os

CONDA_ENV = "/opt/miniconda3/envs/fenicsx-env"
CONDA_PYTHON = f"{CONDA_ENV}/bin/python"

# Prüfe Python-Version und Pfad
print(f"Aktuelles Python: {sys.executable}")
print(f"Python-Version: {sys.version.split()[0]}")

# Wenn nicht in Conda-Umgebung, füge sie zum Pfad hinzu
if CONDA_ENV not in sys.executable:
    conda_site_packages = f"{CONDA_ENV}/lib/python3.12/site-packages"
    if os.path.exists(conda_site_packages) and conda_site_packages not in sys.path:
        sys.path.insert(0, conda_site_packages)
        print(f"✓ Conda-Umgebung zum Python-Pfad hinzugefügt: {conda_site_packages}")
    
    # Setze Bibliothekspfade
    conda_lib = f"{CONDA_ENV}/lib"
    if os.path.exists(conda_lib):
        os.environ["DYLD_LIBRARY_PATH"] = conda_lib + ":" + os.environ.get("DYLD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = conda_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        print(f"✓ Bibliothekspfade gesetzt: {conda_lib}")

# Importiere FEniCSx direkt
try:
    from mpi4py import MPI
    from dolfinx import mesh, fem
    import numpy as np
    
    print("✓ FEniCSx erfolgreich importiert!")
    print(f"✓ MPI Größe: {MPI.COMM_WORLD.size}")
    print(f"✓ Python-Pfad: {len(sys.path)} Einträge")
except ImportError as e:
    print(f"✗ Import-Fehler: {e}")
    print(f"\nVersuche alternativen Import über subprocess...")
    
    # Fallback: Subprocess-Wrapper
    import subprocess
    import json
    
    # Verwende Helper-Skript für bessere Performance
    HELPER_SCRIPT = "/Users/thomashoffbauer/Desktop/Mathematische Texte/fenicsx_helper.py"
    
    def run_fenicsx_code(code):
        """Führt FEniCSx-Code direkt über subprocess aus - vermeidet Blocking"""
        import tempfile
        import time
        import threading
        
        # Konvertiere zu normalen Python-Zahlen (nicht SageMath-Objekte)
        def py_float(x):
            return float(x)
        def py_int(x):
            return int(x)
        
        # Erstelle temporäre Datei für Ergebnis
        result_path = tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.out')
        result_path.close()
        result_path = result_path.name
        
        try:
            # Führe Helper-Skript direkt aus - Code über stdin
            process = subprocess.Popen(
                [HELPER_SCRIPT],
                stdin=subprocess.PIPE,
                stdout=open(result_path, 'w'),
                stderr=subprocess.STDOUT,
                text=True
            )
            
            # Sende Code über stdin (nicht-blockierend)
            process.stdin.write(code)
            process.stdin.close()
            
            # Warte mit Polling - MPI-Import kann 10-30 Sekunden dauern
            max_wait = py_float(90.0)  # Erhöhter Timeout für MPI-Initialisierung
            sleep_interval = py_float(1.0)  # Längeres Intervall
            waited = py_float(0.0)
            
            def check_process_done(p):
                """Prüft ob Prozess fertig ist"""
                result = p.poll()
                if result is None:
                    return None
                return py_int(result) if result is not None else None
            
            # Warte auf Prozess mit Fortschrittsanzeige
            while check_process_done(process) is None and waited < max_wait:
                time.sleep(sleep_interval)
                waited = py_float(waited) + sleep_interval
                waited_int = py_int(waited)
                # Zeige Fortschritt alle 5 Sekunden
                if waited_int > 0 and (waited_int % 5) == 0:
                    print(f"Warte auf FEniCSx (MPI-Import kann lange dauern)... ({waited_int}s)", end='\r')
            
            # Schließe Dateien
            try:
                process.stdin.close()
            except:
                pass
            
            # Warte auf Prozess-Ende
            if check_process_done(process) is None:
                process.terminate()
                time.sleep(py_float(2.0))
                if check_process_done(process) is None:
                    process.kill()
                return {"status": "error", "error": f"Timeout: Prozess hat zu lange gedauert (>{py_int(max_wait)}s). MPI-Import kann beim ersten Mal 30-60 Sekunden dauern."}
            
            # Warte kurz, damit Datei vollständig geschrieben wird
            time.sleep(py_float(1.0))
            
            # Lese Ergebnis
            try:
                if not os.path.exists(result_path):
                    return {"status": "error", "error": "Ergebnisdatei nicht gefunden"}
                
                with open(result_path, 'r') as f:
                    output = f.read()
                
                if "FENICSX_RESULT:" in output:
                    json_str = output.split("FENICSX_RESULT:")[1].strip()
                    # Entferne mögliche zusätzliche Zeilen nach JSON
                    if '\n' in json_str:
                        json_str = json_str.split('\n')[0]
                    return json.loads(json_str)
                else:
                    # Prüfe ob MPI-Fehler vorhanden
                    if "Abort" in output or "MPI" in output:
                        # Versuche trotzdem JSON zu finden
                        lines = output.split('\n')
                        for line in reversed(lines):
                            if "FENICSX_RESULT:" in line:
                                json_str = line.split("FENICSX_RESULT:")[1].strip()
                                return json.loads(json_str)
                    return {"status": "error", "error": f"Kein Ergebnis gefunden. Output (letzte 500 Zeichen): {output[-500:]}"}
            except json.JSONDecodeError as e:
                return {"status": "error", "error": f"JSON-Fehler: {e}. Output (letzte 300 Zeichen): {output[-300:]}"}
            except Exception as e:
                return {"status": "error", "error": f"Fehler beim Lesen: {e}"}
                
        except Exception as e:
            return {"status": "error", "error": str(e)}
        finally:
            # Aufräumen
            try:
                if os.path.exists(result_path):
                    os.unlink(result_path)
            except:
                pass
    
    # Test mit Helper-Skript
    print("Teste FEniCSx über Helper-Skript...")
    print("⚠️  WICHTIG: Der erste MPI-Import kann 30-60 Sekunden dauern!")
    print("   Bitte haben Sie Geduld - nachfolgende Aufrufe sind schneller.\n")
    
    test_code = """print(f"MPI Größe: {MPI.COMM_WORLD.size}")
print("Import erfolgreich!")"""
    
    test_result = run_fenicsx_code(test_code)
    
    if test_result["status"] == "success":
        print("✓ FEniCSx funktioniert über Helper-Skript!")
        if test_result.get("output"):
            print(test_result["output"])
    else:
        print(f"✗ Fehler: {test_result.get('error', 'Unbekannter Fehler')}")
        if "traceback" in test_result:
            print("\nDetaillierter Fehler:")
            print(test_result["traceback"][:500])
        
    # Setze run_fenicsx_code als globale Variable für andere Zellen
    globals()['run_fenicsx_code'] = run_fenicsx_code
    print("\n✓ Wrapper-Funktion 'run_fenicsx_code' ist jetzt verfügbar!")

: 

In [ ]:
# Erstelle ein einfaches Gitter
if 'run_fenicsx_code' in globals():
    code = """domain = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8, mesh.CellType.triangle)
print(f"Gitter erstellt: {domain.geometry.x.shape[0]} Knoten")"""
    result = run_fenicsx_code(code)
    if result["status"] == "success":
        print(result.get("output", ""))
    else:
        print(f"Fehler: {result.get('error', 'Unbekannt')}")
        if "traceback" in result:
            print("\nTraceback:")
            print(result["traceback"][:400])
else:
    print("Bitte führen Sie zuerst die erste Zelle aus!")

In [ ]:
# Definiere Funktionenraum
if 'run_fenicsx_code' in globals():
    code = """from dolfinx.fem import functionspace
domain = mesh.create_unit_square(MPI.COMM_WORLD, 10, 10, mesh.CellType.triangle)
V = functionspace(domain, ("Lagrange", 1))
print(f"Funktionenraum: {V.dofmap.index_map.size_global} Freiheitsgrade")"""
    result = run_fenicsx_code(code)
    if result["status"] == "success":
        print(result.get("output", ""))
    else:
        print(f"Fehler: {result.get('error', 'Unbekannt')}")
        if "traceback" in result:
            print(result["traceback"][:400])
else:
    print("Bitte führen Sie zuerst die erste Zelle aus!")

In [ ]:
# Einfaches Beispiel: Poisson-Gleichung
if 'run_fenicsx_code' in globals():
    code = """from dolfinx import default_scalar_type
from dolfinx.fem import functionspace
from dolfinx.fem.petsc import LinearProblem
from ufl import TrialFunction, TestFunction, dx, grad, inner

domain = mesh.create_unit_square(MPI.COMM_WORLD, 10, 10, mesh.CellType.triangle)
V = functionspace(domain, ("Lagrange", 1))

# Randbedingungen
boundary_facets = mesh.locate_entities_boundary(
    domain, domain.topology.dim - 1, lambda x: np.full(x.shape[1], True)
)
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)
bc = fem.dirichletbc(default_scalar_type(0.0), boundary_dofs, V)

# Bilinearform und Linearform
u = TrialFunction(V)
v = TestFunction(V)
f = fem.Constant(domain, default_scalar_type(1.0))
a = inner(grad(u), grad(v)) * dx
L = f * v * dx

# Löse
problem = LinearProblem(a, L, bcs=[bc])
uh = problem.solve()

print(f"Lösung berechnet!")
print(f"Max: {uh.x.array.max():.4f}, Min: {uh.x.array.min():.4f}")"""
    result = run_fenicsx_code(code)
    if result["status"] == "success":
        print(result.get("output", ""))
    else:
        print(f"Fehler: {result.get('error', 'Unbekannt')}")
        if "traceback" in result:
            print("\nTraceback:")
            print(result["traceback"][:600])
else:
    print("Bitte führen Sie zuerst die erste Zelle aus!")

In [ ]:
# Visualisierung mit pyvista - muss über Wrapper laufen
if 'run_fenicsx_code' in globals():
    code = """from dolfinx import plot
import pyvista
from dolfinx.fem import functionspace
from dolfinx import default_scalar_type
from dolfinx.fem.petsc import LinearProblem
from ufl import TrialFunction, TestFunction, dx, grad, inner

# Erstelle Gitter und löse
domain = mesh.create_unit_square(MPI.COMM_WORLD, 10, 10, mesh.CellType.triangle)
V = functionspace(domain, ("Lagrange", 1))

# Randbedingungen
boundary_facets = mesh.locate_entities_boundary(
    domain, domain.topology.dim - 1, lambda x: np.full(x.shape[1], True)
)
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)
bc = fem.dirichletbc(default_scalar_type(0.0), boundary_dofs, V)

# Löse Poisson-Gleichung
u = TrialFunction(V)
v = TestFunction(V)
f = fem.Constant(domain, default_scalar_type(1.0))
a = inner(grad(u), grad(v)) * dx
L = f * v * dx
problem = LinearProblem(a, L, bcs=[bc])
uh = problem.solve()

# Visualisierung
pyvista.set_jupyter_backend("static")
topology, cell_types, geometry = plot.vtk_mesh(domain, domain.topology.dim)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
grid.point_data["u"] = uh.x.array
grid.set_active_scalars("u")

print("Visualisierung vorbereitet - Grid erstellt")
print(f"Anzahl Punkte: {grid.n_points}, Anzahl Zellen: {grid.n_cells}")
print(f"u-Min: {uh.x.array.min():.4f}, u-Max: {uh.x.array.max():.4f}")
print("Hinweis: Interaktive Visualisierung funktioniert über Wrapper nicht direkt.")
print("Verwenden Sie stattdessen fenicsx_standalone.py für vollständige Visualisierung.")
"""
    result = run_fenicsx_code(code)
    if result["status"] == "success":
        print(result.get("output", ""))
    else:
        print(f"Fehler bei Visualisierung: {result.get('error', 'Unbekannt')}")
        if "traceback" in result:
            print(result["traceback"][:500])
else:
    print("Bitte führen Sie zuerst die erste Zelle aus!")